# 다변수 교호작용 EDA

`is_100` 외의 변수들끼리의 **교호작용(interaction)** 이  
`is_repurchase`, `is_churn_prevented` 에 미치는 영향을 분석합니다.

### 기본 교호작용
| # | 분석 | 설명 |
|---|------|------|
| 1 | gender × age_group | 성별 × 연령대 조합별 재구매/이탈방지율 |
| 2 | gender × payment_device | 성별 × 결제기기 |
| 3 | gender × max_screen | 성별 × 동시접속수 |
| 4 | age_group × payment_device | 연령대 × 결제기기 |
| 5 | age_group × max_screen | 연령대 × 동시접속수 |
| 6 | age_group × billing_method | 연령대 × 결제방식 |
| 7 | is_repurchase × gender | 재구매 여부 × 성별 → 이탈방지율 |
| 8 | is_repurchase × age_group | 재구매 여부 × 연령대 → 이탈방지율 |
| 9 | is_repurchase × payment_device | 재구매 여부 × 결제기기 → 이탈방지율 |

### 심화 교호작용
| # | 분석 | 설명 |
|---|------|------|
| 10 | reg_hour_group × gender | 가입시간대 × 성별 |
| 11 | reg_hour_group × age_group | 가입시간대 × 연령대 |
| 12 | product_code × gender | 상품코드 × 성별 |
| 13 | product_code × age_group | 상품코드 × 연령대 |
| 14 | billing_method × gender | 결제방식 × 성별 |
| 15 | billing_method × payment_device | 결제방식 × 결제기기 |
| 16 | is_promotion × gender | 프로모션 × 성별 |
| 17 | is_promotion × age_group | 프로모션 × 연령대 |
| 18 | is_churn_prevented × gender → 재구매율 | 이탈방지 × 성별 (역방향) |
| 19 | is_churn_prevented × age_group → 재구매율 | 이탈방지 × 연령대 (역방향) |
| 20 | is_churn_prevented × payment_device → 재구매율 | 이탈방지 × 결제기기 (역방향) |
| 21 | duration_group × gender | 구독기간 × 성별 |
| 22 | duration_group × age_group | 구독기간 × 연령대 |
| 23 | duration_group × is_100 | 구독기간 × 100원딜 |
| 24 | 3-way: is_100 × gender × age_group | 3변수 교호작용 |
| 25 | 3-way: is_100 × age_group × payment_device | 3변수 교호작용 |
| 26 | 종합 요약 | 전체 교호작용 효과 크기 비교 |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('../../_data/02_interim/260430_membership_v1(이상치, 이름변경)/Membership_v1.csv', encoding='utf-8-sig')

# 파생 변수
df['is_100'] = (df['price'] == 100).astype(int)
df['price_label'] = df['is_100'].map({1: '100원', 0: '일반요금'})
df['age_group'] = pd.cut(df['age'], bins=[14, 19, 29, 39, 49, 70],
                         labels=['10대', '20대', '30대', '40대', '50대+'])
df['reg_hour_group'] = pd.cut(df['reg_hour'], bins=[-1, 5, 11, 17, 23],
                               labels=['새벽(0-5)', '오전(6-11)', '오후(12-17)', '저녁(18-23)'])
df['repurchase_label'] = df['is_repurchase'].map({0: '재구매 아님', 1: '재구매'})
df['churn_label']     = df['is_churn_prevented'].map({0: '이탈방지 아님', 1: '이탈방지'})
df['promotion_label'] = df['is_promotion'].map({0: '일반', 1: '프로모션'})

# 구독기간 파생 변수
df['reg_date_dt']  = pd.to_datetime(df['reg_date'])
df['end_date_dt']  = pd.to_datetime(df['end_date'])
df['duration_days'] = (df['end_date_dt'] - df['reg_date_dt']).dt.days
df['duration_group'] = pd.cut(df['duration_days'],
                               bins=[-1, 29, 89, 364, 99999],
                               labels=['단기(<30일)', '중기(30-89일)', '장기(90-364일)', '초장기(1년+)'])

OUTCOMES = ['is_repurchase', 'is_churn_prevented']
OUTCOME_LABELS = {'is_repurchase': '재구매율(%)', 'is_churn_prevented': '이탈방지율(%)'}

def two_var_heatmap(df, row_col, col_col, row_label, col_label, figsize=(18, 5)):
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, outcome in zip(axes, OUTCOMES):
        pivot = df.groupby([row_col, col_col], observed=True)[outcome].mean().mul(100).unstack(col_col)
        sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
                    linewidths=0.5, cbar_kws={'label': '%'})
        ax.set_title(f'{row_label} × {col_label} → {OUTCOME_LABELS[outcome]}')
        ax.set_xlabel(col_label)
        ax.set_ylabel(row_label)
    plt.tight_layout()
    plt.show()

def two_var_barplot(df, split_col, split_label, group_col, group_label, outcome, figsize=(18, 5)):
    categories = sorted(df[split_col].dropna().unique().astype(str))
    n = len(categories)
    fig, axes = plt.subplots(1, n, figsize=figsize, sharey=True)
    if n == 1:
        axes = [axes]
    for ax, cat in zip(axes, categories):
        sub = df[df[split_col].astype(str) == cat]
        vals = sub.groupby(group_col, observed=True)[outcome].mean().mul(100).sort_values()
        vals.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
        ax.set_title(f'{split_label} = {cat}\n(n={len(sub):,})')
        ax.set_xlabel('%')
        for p in ax.patches:
            ax.annotate(f'{p.get_width():.1f}', (p.get_width(), p.get_y() + p.get_height()/2),
                        ha='left', va='center', fontsize=8)
    fig.suptitle(f'{split_label} × {group_label} → {OUTCOME_LABELS[outcome]}', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

def single_outcome_heatmap(df, row_col, col_col, row_label, col_label, outcome, figsize=(12, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    pivot = df.groupby([row_col, col_col], observed=True)[outcome].mean().mul(100).unstack(col_col)
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
                linewidths=0.5, cbar_kws={'label': '%'})
    ax.set_title(f'{row_label} × {col_label} → {OUTCOME_LABELS[outcome]}')
    ax.set_xlabel(col_label)
    ax.set_ylabel(row_label)
    plt.tight_layout()
    plt.show()

print('데이터 준비 완료')
print(f'전체 {len(df):,}건')
print(f'구독기간(일) 기술통계:\n{df["duration_days"].describe().round(1)}')

---
# 기본 교호작용 분석

---
## 1. gender × age_group (성별 × 연령대)

In [ ]:
two_var_heatmap(df, 'gender', 'age_group', '성별', '연령대', figsize=(20, 5))

for outcome in OUTCOMES:
    two_var_barplot(df, 'gender', '성별', 'age_group', '연령대', outcome, figsize=(18, 4))

print('=== 성별 × 연령대별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['gender', 'age_group'], observed=True)[OUTCOMES].mean().mul(100).round(2))

---
## 2. gender × payment_device (성별 × 결제기기)

In [ ]:
two_var_heatmap(df, 'gender', 'payment_device', '성별', '결제기기', figsize=(20, 5))

for outcome in OUTCOMES:
    two_var_barplot(df, 'gender', '성별', 'payment_device', '결제기기', outcome, figsize=(18, 4))

print('=== 성별 × 결제기기별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['gender', 'payment_device'])[OUTCOMES].mean().mul(100).round(2))

---
## 3. gender × max_screen (성별 × 동시접속수)

In [ ]:
two_var_heatmap(df, 'gender', 'max_screen', '성별', '동시접속수')

for outcome in OUTCOMES:
    two_var_barplot(df, 'gender', '성별', 'max_screen', '동시접속수', outcome, figsize=(14, 4))

print('=== 성별 × 동시접속수별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['gender', 'max_screen'])[OUTCOMES].mean().mul(100).round(2))

---
## 4. age_group × payment_device (연령대 × 결제기기)

In [ ]:
two_var_heatmap(df, 'age_group', 'payment_device', '연령대', '결제기기', figsize=(22, 5))

for outcome in OUTCOMES:
    two_var_barplot(df, 'age_group', '연령대', 'payment_device', '결제기기', outcome, figsize=(22, 4))

print('=== 연령대 × 결제기기별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['age_group', 'payment_device'], observed=True)[OUTCOMES].mean().mul(100).round(2))

---
## 5. age_group × max_screen (연령대 × 동시접속수)

In [ ]:
two_var_heatmap(df, 'age_group', 'max_screen', '연령대', '동시접속수', figsize=(16, 5))

for outcome in OUTCOMES:
    two_var_barplot(df, 'age_group', '연령대', 'max_screen', '동시접속수', outcome, figsize=(20, 4))

print('=== 연령대 × 동시접속수별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['age_group', 'max_screen'], observed=True)[OUTCOMES].mean().mul(100).round(2))

---
## 6. age_group × billing_method (연령대 × 결제방식)

In [ ]:
two_var_heatmap(df, 'age_group', 'billing_method', '연령대', '결제방식', figsize=(24, 5))

for outcome in OUTCOMES:
    two_var_barplot(df, 'age_group', '연령대', 'billing_method', '결제방식', outcome, figsize=(22, 4))

print('=== 연령대 × 결제방식별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['age_group', 'billing_method'], observed=True)[OUTCOMES].mean().mul(100).round(2))

---
## 7. is_repurchase × gender (재구매 여부 × 성별 → 이탈방지율)

In [ ]:
pivot = df.groupby(['repurchase_label', 'gender'])['is_churn_prevented'].mean().mul(100).unstack('gender')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
pivot.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='white', width=0.7)
axes[0].set_title('재구매 여부 × 성별 → 이탈방지율(%)')
axes[0].set_xlabel('재구매 여부')
axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    h = p.get_height()
    if not np.isnan(h):
        axes[0].annotate(f'{h:.1f}', (p.get_x() + p.get_width()/2, h),
                         ha='center', va='bottom', fontsize=8)
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1],
            linewidths=0.5, cbar_kws={'label': '%'})
axes[1].set_title('히트맵: 재구매 × 성별 → 이탈방지율(%)')
plt.tight_layout()
plt.show()

print('=== 재구매여부 × 성별별 이탈방지율 (%) ===')
print(pivot.round(2))

---
## 8. is_repurchase × age_group (재구매 여부 × 연령대 → 이탈방지율)

In [ ]:
pivot = df.groupby(['repurchase_label', 'age_group'], observed=True)['is_churn_prevented'].mean().mul(100).unstack('age_group')

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
pivot.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='white', width=0.7)
axes[0].set_title('재구매 여부 × 연령대 → 이탈방지율(%)')
axes[0].set_xlabel('재구매 여부')
axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    h = p.get_height()
    if not np.isnan(h):
        axes[0].annotate(f'{h:.1f}', (p.get_x() + p.get_width()/2, h),
                         ha='center', va='bottom', fontsize=7.5)
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1],
            linewidths=0.5, cbar_kws={'label': '%'})
axes[1].set_title('히트맵: 재구매 × 연령대 → 이탈방지율(%)')
plt.tight_layout()
plt.show()

print('=== 재구매여부 × 연령대별 이탈방지율 (%) ===')
print(pivot.round(2))

---
## 9. is_repurchase × payment_device (재구매 여부 × 결제기기 → 이탈방지율)

In [ ]:
pivot = df.groupby(['repurchase_label', 'payment_device'])['is_churn_prevented'].mean().mul(100).unstack('payment_device')

fig, axes = plt.subplots(1, 2, figsize=(20, 5))
pivot.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='white', width=0.7)
axes[0].set_title('재구매 여부 × 결제기기 → 이탈방지율(%)')
axes[0].set_xlabel('재구매 여부')
axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='결제기기', fontsize=8)
for p in axes[0].patches:
    h = p.get_height()
    if not np.isnan(h):
        axes[0].annotate(f'{h:.1f}', (p.get_x() + p.get_width()/2, h),
                         ha='center', va='bottom', fontsize=7)
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1],
            linewidths=0.5, cbar_kws={'label': '%'})
axes[1].set_title('히트맵: 재구매 × 결제기기 → 이탈방지율(%)')
plt.tight_layout()
plt.show()

print('=== 재구매여부 × 결제기기별 이탈방지율 (%) ===')
print(pivot.round(2))

---
# 심화 교호작용 분석

---
## 10. reg_hour_group × gender (가입시간대 × 성별)

In [ ]:
two_var_heatmap(df, 'reg_hour_group', 'gender', '가입시간대', '성별', figsize=(16, 5))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, outcome in zip(axes, OUTCOMES):
    for gender, sub in df.groupby('gender'):
        trend = sub.groupby('reg_hour_group', observed=True)[outcome].mean().mul(100)
        ax.plot(trend.index.astype(str), trend.values, marker='o', label=f'gender={gender}')
    ax.set_title(f'가입시간대별 {OUTCOME_LABELS[outcome]} (성별 구분)')
    ax.set_ylabel('%')
    ax.legend()
plt.tight_layout()
plt.show()

print('=== 가입시간대 × 성별별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['reg_hour_group', 'gender'], observed=True)[OUTCOMES].mean().mul(100).round(2))

---
## 11. reg_hour_group × age_group (가입시간대 × 연령대)

In [ ]:
two_var_heatmap(df, 'reg_hour_group', 'age_group', '가입시간대', '연령대', figsize=(20, 5))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, outcome in zip(axes, OUTCOMES):
    for age, sub in df.groupby('age_group', observed=True):
        trend = sub.groupby('reg_hour_group', observed=True)[outcome].mean().mul(100)
        ax.plot(trend.index.astype(str), trend.values, marker='o', label=str(age))
    ax.set_title(f'가입시간대별 {OUTCOME_LABELS[outcome]} (연령대 구분)')
    ax.set_ylabel('%')
    ax.legend(title='연령대')
plt.tight_layout()
plt.show()

print('=== 가입시간대 × 연령대별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['reg_hour_group', 'age_group'], observed=True)[OUTCOMES].mean().mul(100).round(2))

---
## 12. product_code × gender (상품코드 × 성별)

In [ ]:
two_var_heatmap(df, 'product_code', 'gender', '상품코드', '성별', figsize=(14, 8))

print('=== 상품코드 × 성별별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['product_code', 'gender'])[OUTCOMES].mean().mul(100).round(2))
print('\n=== 상품코드 × 성별 샘플 수 ===')
print(df.groupby(['product_code', 'gender']).size().unstack('gender').fillna(0).astype(int))

---
## 13. product_code × age_group (상품코드 × 연령대)

In [ ]:
two_var_heatmap(df, 'product_code', 'age_group', '상품코드', '연령대', figsize=(22, 8))

print('=== 상품코드 × 연령대별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['product_code', 'age_group'], observed=True)[OUTCOMES].mean().mul(100).round(2))
print('\n=== 상품코드 × 연령대 샘플 수 ===')
print(df.groupby(['product_code', 'age_group'], observed=True).size().unstack('age_group').fillna(0).astype(int))

---
## 14. billing_method × gender (결제방식 × 성별)

In [ ]:
two_var_heatmap(df, 'billing_method', 'gender', '결제방식', '성별', figsize=(14, 8))

print('=== 결제방식 × 성별별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['billing_method', 'gender'])[OUTCOMES].mean().mul(100).round(2))
print('\n=== 결제방식 × 성별 샘플 수 ===')
print(df.groupby(['billing_method', 'gender']).size().unstack('gender').fillna(0).astype(int))

---
## 15. billing_method × payment_device (결제방식 × 결제기기)

In [ ]:
two_var_heatmap(df, 'billing_method', 'payment_device', '결제방식', '결제기기', figsize=(22, 8))

print('=== 결제방식 × 결제기기별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['billing_method', 'payment_device'])[OUTCOMES].mean().mul(100).round(2))
print('\n=== 결제방식 × 결제기기 샘플 수 ===')
print(df.groupby(['billing_method', 'payment_device']).size().unstack('payment_device').fillna(0).astype(int))

---
## 16. is_promotion × gender (프로모션 × 성별)

In [ ]:
two_var_heatmap(df, 'promotion_label', 'gender', '프로모션여부', '성별', figsize=(14, 5))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, outcome in zip(axes, OUTCOMES):
    pivot = df.groupby(['promotion_label', 'gender'])[outcome].mean().mul(100).unstack('gender')
    pivot.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white', width=0.7)
    ax.set_title(f'프로모션 × 성별 → {OUTCOME_LABELS[outcome]}')
    ax.set_xlabel('프로모션여부')
    ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=0)
    for p in ax.patches:
        h = p.get_height()
        if not np.isnan(h):
            ax.annotate(f'{h:.1f}', (p.get_x() + p.get_width()/2, h),
                        ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

print('=== 프로모션 × 성별별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['promotion_label', 'gender'])[OUTCOMES].mean().mul(100).round(2))

---
## 17. is_promotion × age_group (프로모션 × 연령대)

In [ ]:
two_var_heatmap(df, 'promotion_label', 'age_group', '프로모션여부', '연령대', figsize=(20, 5))

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
for ax, outcome in zip(axes, OUTCOMES):
    pivot = df.groupby(['promotion_label', 'age_group'], observed=True)[outcome].mean().mul(100).unstack('age_group')
    pivot.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white', width=0.7)
    ax.set_title(f'프로모션 × 연령대 → {OUTCOME_LABELS[outcome]}')
    ax.set_xlabel('프로모션여부')
    ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=0)
    for p in ax.patches:
        h = p.get_height()
        if not np.isnan(h):
            ax.annotate(f'{h:.1f}', (p.get_x() + p.get_width()/2, h),
                        ha='center', va='bottom', fontsize=7.5)
plt.tight_layout()
plt.show()

print('=== 프로모션 × 연령대별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['promotion_label', 'age_group'], observed=True)[OUTCOMES].mean().mul(100).round(2))

---
## 18. is_churn_prevented × gender → 재구매율 (역방향)
> 7~9번은 `재구매 → 이탈방지` 방향, 여기서는 반대 방향

In [ ]:
pivot = df.groupby(['churn_label', 'gender'])['is_repurchase'].mean().mul(100).unstack('gender')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
pivot.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='white', width=0.7)
axes[0].set_title('이탈방지 여부 × 성별 → 재구매율(%)')
axes[0].set_xlabel('이탈방지 여부')
axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    h = p.get_height()
    if not np.isnan(h):
        axes[0].annotate(f'{h:.1f}', (p.get_x() + p.get_width()/2, h),
                         ha='center', va='bottom', fontsize=8)
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1],
            linewidths=0.5, cbar_kws={'label': '%'})
axes[1].set_title('히트맵: 이탈방지 × 성별 → 재구매율(%)')
plt.tight_layout()
plt.show()

print('=== 이탈방지여부 × 성별별 재구매율 (%) ===')
print(pivot.round(2))

---
## 19. is_churn_prevented × age_group → 재구매율 (역방향)

In [ ]:
pivot = df.groupby(['churn_label', 'age_group'], observed=True)['is_repurchase'].mean().mul(100).unstack('age_group')

fig, axes = plt.subplots(1, 2, figsize=(20, 5))
pivot.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='white', width=0.7)
axes[0].set_title('이탈방지 여부 × 연령대 → 재구매율(%)')
axes[0].set_xlabel('이탈방지 여부')
axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    h = p.get_height()
    if not np.isnan(h):
        axes[0].annotate(f'{h:.1f}', (p.get_x() + p.get_width()/2, h),
                         ha='center', va='bottom', fontsize=7.5)
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1],
            linewidths=0.5, cbar_kws={'label': '%'})
axes[1].set_title('히트맵: 이탈방지 × 연령대 → 재구매율(%)')
plt.tight_layout()
plt.show()

print('=== 이탈방지여부 × 연령대별 재구매율 (%) ===')
print(pivot.round(2))

---
## 20. is_churn_prevented × payment_device → 재구매율 (역방향)

In [ ]:
pivot = df.groupby(['churn_label', 'payment_device'])['is_repurchase'].mean().mul(100).unstack('payment_device')

fig, axes = plt.subplots(1, 2, figsize=(20, 5))
pivot.plot(kind='bar', ax=axes[0], colormap='Set2', edgecolor='white', width=0.7)
axes[0].set_title('이탈방지 여부 × 결제기기 → 재구매율(%)')
axes[0].set_xlabel('이탈방지 여부')
axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='결제기기', fontsize=8)
for p in axes[0].patches:
    h = p.get_height()
    if not np.isnan(h):
        axes[0].annotate(f'{h:.1f}', (p.get_x() + p.get_width()/2, h),
                         ha='center', va='bottom', fontsize=7)
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1],
            linewidths=0.5, cbar_kws={'label': '%'})
axes[1].set_title('히트맵: 이탈방지 × 결제기기 → 재구매율(%)')
plt.tight_layout()
plt.show()

print('=== 이탈방지여부 × 결제기기별 재구매율 (%) ===')
print(pivot.round(2))

---
## 21. duration_group × gender (구독기간 × 성별)
> 구독기간 = `end_date - reg_date` 로 계산한 파생 변수

In [ ]:
print('=== 구독기간 분포 ===')
print(df['duration_group'].value_counts().sort_index())

two_var_heatmap(df, 'duration_group', 'gender', '구독기간', '성별', figsize=(16, 5))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, outcome in zip(axes, OUTCOMES):
    pivot = df.groupby(['duration_group', 'gender'], observed=True)[outcome].mean().mul(100).unstack('gender')
    pivot.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white', width=0.7)
    ax.set_title(f'구독기간 × 성별 → {OUTCOME_LABELS[outcome]}')
    ax.set_xlabel('구독기간')
    ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=30)
    for p in ax.patches:
        h = p.get_height()
        if not np.isnan(h):
            ax.annotate(f'{h:.1f}', (p.get_x() + p.get_width()/2, h),
                        ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

print('=== 구독기간 × 성별별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['duration_group', 'gender'], observed=True)[OUTCOMES].mean().mul(100).round(2))

---
## 22. duration_group × age_group (구독기간 × 연령대)

In [ ]:
two_var_heatmap(df, 'duration_group', 'age_group', '구독기간', '연령대', figsize=(22, 5))

fig, axes = plt.subplots(1, 2, figsize=(20, 5))
for ax, outcome in zip(axes, OUTCOMES):
    pivot = df.groupby(['duration_group', 'age_group'], observed=True)[outcome].mean().mul(100).unstack('age_group')
    pivot.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white', width=0.7)
    ax.set_title(f'구독기간 × 연령대 → {OUTCOME_LABELS[outcome]}')
    ax.set_xlabel('구독기간')
    ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=30)
    for p in ax.patches:
        h = p.get_height()
        if not np.isnan(h):
            ax.annotate(f'{h:.1f}', (p.get_x() + p.get_width()/2, h),
                        ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.show()

print('=== 구독기간 × 연령대별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['duration_group', 'age_group'], observed=True)[OUTCOMES].mean().mul(100).round(2))

---
## 23. duration_group × is_100 (구독기간 × 100원딜)
> 100원딜 사용자와 일반 사용자의 구독기간 패턴 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (label, sub) in zip(axes, [('100원', df[df['is_100']==1]), ('일반요금', df[df['is_100']==0])]):
    cnt = sub['duration_group'].value_counts().sort_index()
    cnt.plot(kind='bar', ax=ax, color='steelblue' if label == '100원' else 'salmon', edgecolor='white')
    ax.set_title(f'구독기간 분포 [{label}]')
    ax.set_xlabel('구독기간')
    ax.set_ylabel('count')
    ax.tick_params(axis='x', rotation=30)
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()),
                    ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

two_var_heatmap(df, 'duration_group', 'price_label', '구독기간', '가격구분', figsize=(14, 5))

print('=== 구독기간(일) 기술통계: 100원 vs 일반요금 ===')
print(df.groupby('price_label')['duration_days'].describe().round(1))

print('\n=== 구독기간 × 100원딜별 재구매율/이탈방지율 (%) ===')
print(df.groupby(['duration_group', 'price_label'], observed=True)[OUTCOMES].mean().mul(100).round(2))

---
## 24. 3-way: is_100 × gender × age_group → 재구매율/이탈방지율
> 3변수 교호작용: 100원딜 여부 + 성별 + 연령대가 결합될 때 어떻게 달라지는가

In [ ]:
genders = sorted(df['gender'].dropna().unique())
price_labels = ['100원', '일반요금']

for outcome in OUTCOMES:
    fig, axes = plt.subplots(len(price_labels), len(genders),
                             figsize=(5 * len(genders), 8), sharey=True)
    for i, price in enumerate(price_labels):
        for j, gender in enumerate(genders):
            ax = axes[i][j]
            sub = df[(df['price_label'] == price) & (df['gender'] == gender)]
            vals = sub.groupby('age_group', observed=True)[outcome].mean().mul(100)
            color = 'steelblue' if price == '100원' else 'salmon'
            bars = ax.bar(vals.index.astype(str), vals.values, color=color, edgecolor='white')
            ax.set_title(f'{price} / gender={gender}\n(n={len(sub):,})', fontsize=9)
            ax.set_ylim(0, 100)
            ax.set_xlabel('연령대', fontsize=8)
            if j == 0:
                ax.set_ylabel(OUTCOME_LABELS[outcome], fontsize=8)
            ax.tick_params(axis='x', rotation=30, labelsize=7)
            for bar in bars:
                h = bar.get_height()
                if not np.isnan(h) and h > 0:
                    ax.annotate(f'{h:.1f}', (bar.get_x() + bar.get_width()/2, h),
                                ha='center', va='bottom', fontsize=7)
    fig.suptitle(f'3-way: is_100 × gender × age_group → {OUTCOME_LABELS[outcome]}',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

print('=== 3-way 수치표: is_100 × gender × age_group (%) ===')
print(df.groupby(['price_label', 'gender', 'age_group'], observed=True)[OUTCOMES].mean().mul(100).round(2).to_string())

---
## 25. 3-way: is_100 × age_group × payment_device → 재구매율/이탈방지율

In [ ]:
age_groups = df['age_group'].cat.categories.tolist()

for outcome in OUTCOMES:
    fig, axes = plt.subplots(len(price_labels), len(age_groups),
                             figsize=(4 * len(age_groups), 8), sharey=True)
    for i, price in enumerate(price_labels):
        for j, age in enumerate(age_groups):
            ax = axes[i][j]
            sub = df[(df['price_label'] == price) & (df['age_group'] == age)]
            vals = sub.groupby('payment_device')[outcome].mean().mul(100).sort_values()
            color = 'steelblue' if price == '100원' else 'salmon'
            bars = ax.barh(vals.index.astype(str), vals.values, color=color, edgecolor='white')
            ax.set_title(f'{price} / {age}\n(n={len(sub):,})', fontsize=8)
            ax.set_xlim(0, 100)
            if i == len(price_labels) - 1:
                ax.set_xlabel('%', fontsize=7)
            ax.tick_params(axis='y', labelsize=7)
            for bar in bars:
                w = bar.get_width()
                if not np.isnan(w) and w > 0:
                    ax.annotate(f'{w:.1f}', (w, bar.get_y() + bar.get_height()/2),
                                ha='left', va='center', fontsize=6.5)
    fig.suptitle(f'3-way: is_100 × age_group × payment_device → {OUTCOME_LABELS[outcome]}',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

print('=== 3-way 수치표: is_100 × age_group × payment_device (%) ===')
print(df.groupby(['price_label', 'age_group', 'payment_device'], observed=True)[OUTCOMES].mean().mul(100).round(2).to_string())

---
## 26. 종합 요약: 전체 교호작용 효과 크기 비교
> 각 변수 조합의 효과 범위(max - min)를 비교해 어떤 교호작용이 가장 강한지 확인

In [ ]:
all_pairs = [
    # 기본 교호작용
    ('gender',          'age_group',        '성별 × 연령대'),
    ('gender',          'payment_device',   '성별 × 결제기기'),
    ('gender',          'max_screen',       '성별 × 동시접속수'),
    ('age_group',       'payment_device',   '연령대 × 결제기기'),
    ('age_group',       'max_screen',       '연령대 × 동시접속수'),
    ('age_group',       'billing_method',   '연령대 × 결제방식'),
    ('repurchase_label','gender',           '재구매 × 성별 → 이탈방지'),
    ('repurchase_label','age_group',        '재구매 × 연령대 → 이탈방지'),
    ('repurchase_label','payment_device',   '재구매 × 결제기기 → 이탈방지'),
    # 심화 교호작용
    ('reg_hour_group',  'gender',           '가입시간대 × 성별'),
    ('reg_hour_group',  'age_group',        '가입시간대 × 연령대'),
    ('product_code',    'gender',           '상품코드 × 성별'),
    ('product_code',    'age_group',        '상품코드 × 연령대'),
    ('billing_method',  'gender',           '결제방식 × 성별'),
    ('billing_method',  'payment_device',   '결제방식 × 결제기기'),
    ('promotion_label', 'gender',           '프로모션 × 성별'),
    ('promotion_label', 'age_group',        '프로모션 × 연령대'),
    ('churn_label',     'gender',           '이탈방지 × 성별 → 재구매'),
    ('churn_label',     'age_group',        '이탈방지 × 연령대 → 재구매'),
    ('churn_label',     'payment_device',   '이탈방지 × 결제기기 → 재구매'),
    ('duration_group',  'gender',           '구독기간 × 성별'),
    ('duration_group',  'age_group',        '구독기간 × 연령대'),
    ('duration_group',  'price_label',      '구독기간 × 100원딜'),
]

for outcome in OUTCOMES:
    rows = []
    for r_col, c_col, label in all_pairs:
        try:
            vals = df.groupby([r_col, c_col], observed=True)[outcome].mean().mul(100).dropna()
            rows.append({
                '교호작용': label,
                '최소값(%)': round(vals.min(), 2),
                '최대값(%)': round(vals.max(), 2),
                '효과범위(max-min)': round(vals.max() - vals.min(), 2),
                '표준편차': round(vals.std(), 2)
            })
        except Exception:
            pass
    summary = pd.DataFrame(rows).set_index('교호작용').sort_values('효과범위(max-min)', ascending=False)
    print(f'\n===== {OUTCOME_LABELS[outcome]} — 교호작용 효과 크기 순위 =====')
    print(summary.to_string())

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.barh(summary.index, summary['효과범위(max-min)'], color='steelblue', edgecolor='white')
    ax.set_title(f'{OUTCOME_LABELS[outcome]} — 교호작용 효과 범위 순위 (max - min)')
    ax.set_xlabel('효과 범위 (%p)')
    ax.axvline(summary['효과범위(max-min)'].mean(), color='red', linestyle='--', label='평균')
    ax.legend()
    for p in ax.patches:
        ax.annotate(f'{p.get_width():.1f}', (p.get_width(), p.get_y() + p.get_height()/2),
                    ha='left', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()